# FLUX Q8 单卡 vs 双开 A/B 测试 v2
同会话、同条件：先单跑 1 张，再双开 2 张，对比每图成本。T5 走本地拷贝/下载，绕开挂载盘慢读。

In [ ]:
# ===== 参数区 =====
SINGLE = [("G.E.M. Deng Ziqi, famous Chinese pop singer, singing passionately on a beach at golden sunset, holding a microphone, ocean waves and palm trees in the background, warm golden hour light, photorealistic portrait, 85mm lens, ultra detailed", 42)]
DUAL = [
    ("G.E.M. Deng Ziqi, famous Chinese pop singer, performing live on a concert stage, holding a microphone, dramatic purple and blue stage lights, cheering crowd silhouettes, photorealistic, ultra detailed", 43),
    ("G.E.M. Deng Ziqi, famous Chinese pop singer, singing in a recording studio, holding headphones, warm wooden interior, studio microphone with pop filter, photorealistic portrait, ultra detailed", 44),
]
STEPS, GUIDANCE, W, H = 28, 3.5, 1024, 1024
STAGGER_S = 60

In [ ]:
# ===== 环境与路径 =====
import subprocess, sys, glob, os, time, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf==0.19.0"], check=True)
import gguf  # 先于 diffusers
import torch
cands = glob.glob("/kaggle/input/datasets/**/*.gguf", recursive=True) + glob.glob("/kaggle/input/**/*.gguf", recursive=True)
assert cands, "数据集未挂载"
GGUF_PATH = cands[0]; DS = os.path.dirname(GGUF_PATH)
print("GGUF:", GGUF_PATH); print("DS:", DS); print(sorted(os.listdir(DS)))
print("GPUs:", torch.cuda.device_count(), "| RAM GB:", round(os.sysconf("SC_PAGE_SIZE")*os.sysconf("SC_PHYS_PAGES")/2**30,1))

In [ ]:
# ===== T5 本地化：优先从挂载盘顺序拷贝 text_encoder_2（掐表）；超时则 HF 下载大文件 =====
import shutil
SRC = os.path.join(DS, "text_encoder_2")
DST = "/kaggle/working/ds_local/text_encoder_2"
os.makedirs(DST, exist_ok=True)
big = [f for f in glob.glob(SRC + "/*.safetensors")]
print("T5 源文件:", big)
t0 = time.time(); copied = True
try:
    for f in os.listdir(SRC):
        shutil.copy2(os.path.join(SRC, f), os.path.join(DST, f))
except Exception as e:
    copied = False; print("copy error:", e)
cp_s = time.time() - t0
print(f"cp 用时 {cp_s:.1f}s, copied={copied}")
if (not copied) or cp_s > 240:
    print("拷贝过慢/失败 -> 改走 HF 下载大文件")
    from huggingface_hub import hf_hub_download
    tgt = os.path.basename(big[0]) if big else "model.safetensors"
    p = hf_hub_download("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", local_dir="/kaggle/working/hf_t5")
    shutil.copy2(p, os.path.join(DST, tgt))
    print("HF 下载完成 ->", os.path.join(DST, tgt))
print(sorted(os.listdir(DST)))

In [ ]:
# ===== 编码 3 条提示词 =====
from transformers import CLIPTextModel, CLIPTokenizer, T5EncoderModel, T5TokenizerFast
tok1 = CLIPTokenizer.from_pretrained(DS, subfolder="tokenizer")
tok2 = T5TokenizerFast.from_pretrained(DS, subfolder="tokenizer_2")
t0=time.time()
te1 = CLIPTextModel.from_pretrained(DS, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda:0")
te2 = T5EncoderModel.from_pretrained("/kaggle/working/ds_local/text_encoder_2", torch_dtype=torch.float16).to("cuda:0")
print(f"编码器加载 {time.time()-t0:.1f}s")
torch.set_grad_enabled(False)  # 实测修复：编码开梯度会 OOM
def encode(prompt):
    i1 = tok1(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    pooled = te1(i1).pooler_output
    i2 = tok2(prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt").input_ids.to("cuda:0")
    return te2(i2).last_hidden_state.cpu(), pooled.cpu()
os.makedirs("/kaggle/working/embeds", exist_ok=True)
ALL = SINGLE + DUAL
for i,(p,s) in enumerate(ALL):
    emb, pooled = encode(p)
    torch.save({"emb": emb, "pooled": pooled, "seed": s}, f"/kaggle/working/embeds/emb_{i}.pt")
    print(i, emb.shape)
del te1, te2; torch.cuda.empty_cache()
print("编码完成")

In [ ]:
# ===== worker 脚本（GGUFQuantizationConfig 为实测正确写法）=====
worker = r"""
import sys, time, json
import gguf
import torch
from diffusers import FluxTransformer2DModel, FluxPipeline, AutoencoderKL, FlowMatchEulerDiscreteScheduler, GGUFQuantizationConfig
idx = int(sys.argv[1]); ds = sys.argv[2]; gguf_path = sys.argv[3]
t0 = time.time()
transformer = FluxTransformer2DModel.from_single_file(gguf_path, config=f"{ds}/transformer", quantization_config=GGUFQuantizationConfig(compute_dtype=torch.float16))
transformer = transformer.to("cuda:0")
vae = AutoencoderKL.from_pretrained(ds, subfolder="vae", torch_dtype=torch.float16).to("cuda:0")
sched = FlowMatchEulerDiscreteScheduler.from_pretrained(ds, subfolder="scheduler")
pipe = FluxPipeline(scheduler=sched, vae=vae, text_encoder=None, tokenizer=None, text_encoder_2=None, tokenizer_2=None, transformer=transformer)
t_load = time.time() - t0
d = torch.load(f"/kaggle/working/embeds/emb_{idx}.pt")
g = torch.Generator(device="cuda:0").manual_seed(int(d["seed"]))
t1 = time.time()
img = pipe(prompt_embeds=d["emb"].to("cuda:0", torch.float16), pooled_prompt_embeds=d["pooled"].to("cuda:0", torch.float16),
           width=1024, height=1024, num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
t_gen = time.time() - t1
img.save(f"/kaggle/working/out_{idx}.png")
print(json.dumps({"idx": idx, "load_s": round(t_load,1), "gen_plus_decode_s": round(t_gen,1), "peak_gb": round(torch.cuda.max_memory_allocated()/2**30,2)}))
"""
open("/kaggle/working/worker.py","w").write(worker)
print("worker.py 已写入")

In [ ]:
# ===== A：单卡单张（emb 0）=====
import subprocess
t0 = time.time()
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
r = subprocess.run([sys.executable, "/kaggle/working/worker.py", "0", DS, GGUF_PATH], env=env, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-2000:] if r.returncode else "")
print(f"SINGLE 总耗时(进程墙钟): {time.time()-t0:.1f}s")

In [ ]:
# ===== B：双卡双开（emb 1,2，错峰 60s）=====
procs = []
t0 = time.time()
for k, idx in enumerate([1, 2]):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    procs.append(subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(idx), DS, GGUF_PATH], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True))
    if k == 0: time.sleep(STAGGER_S)
for p in procs: print(p.communicate()[0])
wall = time.time() - t0
print(f"DUAL 墙钟: {wall:.1f}s | 每图: {wall/2:.1f}s")

In [ ]:
from IPython.display import Image, display
for i in range(3): display(Image(filename=f"/kaggle/working/out_{i}.png"))